In [ ]:
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForVision2Seq, Blip2ForConditionalGeneration, Blip2Processor
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
import re


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
MODEL_NAME = "llava-hf/llava-v1.6-mistral-7b-hf"
MODEL_TYPE = "llava"

# MODEL_NAME = "Salesforce/instructblip-vicuna-7b"
# MODEL_TYPE = "instructblip"

# MODEL_NAME = "Salesforce/blip2-opt-2.7b"
# MODEL_TYPE = "blip2"


In [ ]:
if MODEL_TYPE == "blip2":
    from transformers import Blip2Processor, Blip2ForConditionalGeneration
    processor = Blip2Processor.from_pretrained(MODEL_NAME)
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
elif MODEL_TYPE == "instructblip":
    from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
    processor = InstructBlipProcessor.from_pretrained(MODEL_NAME)
    model = InstructBlipForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
elif MODEL_TYPE == "llava":
    from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
    processor = LlavaNextProcessor.from_pretrained(MODEL_NAME)
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )

if not torch.cuda.is_available():
    model = model.to(device)

model.eval()

In [ ]:
dataset = load_dataset("derek-thomas/ScienceQA")
test_dataset = dataset['test']

In [ ]:
import warnings
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()
warnings.filterwarnings('ignore')

In [ ]:
# ==================== HELPER FUNCTIONS ====================

def format_choices(choices):
    """Format choices as A) B) C) D)"""
    return "\n".join([f"{chr(65+i)}) {choice}" for i, choice in enumerate(choices)])

def create_prompt(question, choices, hint=""):
    """Create a prompt for the VLM"""
    choices_text = format_choices(choices)

    if hint and hint.strip():
        if len(hint) > 500:
            hint = hint[:497] + "..."
        prompt = f"""Context: {hint}

Question: {question}

Choices:
{choices_text}

Answer with only the letter (A, B, C, or D) of the correct choice."""
    else:
        prompt = f"""Question: {question}

Choices:
{choices_text}

Answer with only the letter (A, B, C, or D) of the correct choice."""

    return prompt

def extract_answer(response, num_choices):
    """Extract the predicted choice index from model response"""
    response = response.strip().upper()

    patterns = [
        r'\b([A-Z])\)',
        r'\b([A-Z])\.',
        r'\(([A-Z])\)',
        r'^([A-Z])\b',
        r'\b([A-Z])\b',
    ]

    for pattern in patterns:
        match = re.search(pattern, response)
        if match:
            letter = match.group(1)
            idx = ord(letter) - ord('A')
            if 0 <= idx < num_choices:
                return idx

    return -1

def predict_single(question, choices, hint, image):
    """
    Predict answer for a single question.
    The VLM sees both the image (if available) and the text prompt.
    """
    prompt = create_prompt(question, choices, hint)

    try:
        if image is not None:
            if not isinstance(image, Image.Image):
                if hasattr(image, 'convert'):
                    image = image
                else:
                    image = Image.fromarray(np.array(image))

            if image.mode != 'RGB':
                image = image.convert('RGB')

            if MODEL_TYPE == "llava":
                prompt = f"<image>\n{prompt}"

            inputs = processor(
                images=image,
                text=prompt,
                return_tensors="pt",
                padding=True
            ).to(device, torch.float16 if torch.cuda.is_available() else torch.float32)
        else:
            inputs = processor(
                text=prompt,
                return_tensors="pt",
                padding=True
            ).to(device, torch.float16 if torch.cuda.is_available() else torch.float32)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=10,
                num_beams=1,
                do_sample=False,
                pad_token_id=processor.tokenizer.eos_token_id,
            )

        response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        pred_idx = extract_answer(response, len(choices))

        return pred_idx, response

    except Exception as e:
        print(f"\nError processing question: {str(e)[:100]}")
        return -1, f"ERROR: {str(e)[:50]}"


In [ ]:
# ==================== EVALUATION WITH BREAKDOWN ====================

def evaluate_with_breakdown(dataset, max_samples=None):
    """Evaluate VLM and break down by subject, context, and grade"""

    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))
        print(f"Testing on {len(dataset)} samples")

    all_predictions = []
    all_labels = []
    all_subjects = []
    all_has_image = []
    all_has_text = []
    all_grades = []
    all_responses = []

    for i in tqdm(range(len(dataset)), desc="Evaluating"):
        example = dataset[i]

        question = example['question']
        choices = example['choices']
        hint = example['hint'] if example['hint'] else ""
        answer = example['answer']
        image = example['image']
        subject = example['subject']
        grade = example['grade']

        pred_idx, response = predict_single(question, choices, hint, image)

        all_predictions.append(pred_idx)
        all_labels.append(answer)
        all_subjects.append(subject)
        all_grades.append(grade)
        all_responses.append(response)

        all_has_image.append(image is not None)
        all_has_text.append(hint != '' and hint is not None)

    predictions = np.array(all_predictions)
    labels = np.array(all_labels)
    subjects = np.array(all_subjects)
    has_image = np.array(all_has_image)
    has_text = np.array(all_has_text)
    grades = np.array(all_grades)

    results = {}

    valid_mask = predictions >= 0
    overall_acc = (predictions[valid_mask] == labels[valid_mask]).mean() * 100
    results['Overall'] = overall_acc
    results['Valid_predictions'] = valid_mask.sum() / len(predictions) * 100

    for subject in ['natural science', 'social science', 'language science']:
        mask = (subjects == subject) & valid_mask
        if mask.sum() > 0:
            acc = (predictions[mask] == labels[mask]).mean() * 100
            results[subject.upper()[:3]] = acc

    mask = has_text & ~has_image & valid_mask
    if mask.sum() > 0:
        results['TXT'] = (predictions[mask] == labels[mask]).mean() * 100

    mask = has_image & ~has_text & valid_mask
    if mask.sum() > 0:
        results['IMG'] = (predictions[mask] == labels[mask]).mean() * 100

    mask = ~has_image & ~has_text & valid_mask
    if mask.sum() > 0:
        results['NO'] = (predictions[mask] == labels[mask]).mean() * 100

    mask = has_image & has_text & valid_mask
    if mask.sum() > 0:
        results['TXT+IMG'] = (predictions[mask] == labels[mask]).mean() * 100

    grade_nums = []
    for g in grades:
        try:
            grade_nums.append(int(g.replace('grade', '')))
        except:
            grade_nums.append(0)
    grade_nums = np.array(grade_nums)

    mask = (grade_nums >= 1) & (grade_nums <= 6) & valid_mask
    if mask.sum() > 0:
        results['G1-6'] = (predictions[mask] == labels[mask]).mean() * 100

    mask = (grade_nums >= 7) & (grade_nums <= 12) & valid_mask
    if mask.sum() > 0:
        results['G7-12'] = (predictions[mask] == labels[mask]).mean() * 100

    return results, all_responses, predictions, labels


In [ ]:
results, responses, preds, labels = evaluate_with_breakdown(test_dataset)

In [ ]:
# ==================== PRINT AND VISUALIZE RESULTS ====================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("\n" + "="*60)
print("EVALUATION RESULTS - ScienceQA Test Set")
print("="*60)

print("\nOVERALL PERFORMANCE:")
print("-" * 40)
print(f"Overall Accuracy:      {results['Overall']:.2f}%")
print(f"Valid Predictions:     {results['Valid_predictions']:.2f}%")
print(f"Failed Extractions:    {100 - results['Valid_predictions']:.2f}%")

print("\nACCURACY BY SUBJECT:")
print("-" * 40)
subject_results = {
    'Natural Science': results.get('NAT', 0),
    'Social Science': results.get('SOC', 0),
    'Language Science': results.get('LAN', 0)
}
for subject, acc in subject_results.items():
    if acc > 0:
        print(f"{subject:20s}: {acc:6.2f}%")

print("\nACCURACY BY CONTEXT TYPE:")
print("-" * 40)
context_results = {
    'Text Only': results.get('TXT', 0),
    'Image Only': results.get('IMG', 0),
    'No Context': results.get('NO', 0),
    'Text + Image': results.get('TXT+IMG', 0)
}
for context, acc in context_results.items():
    if acc > 0:
        bar = '|' * int(acc / 2)
        print(f"{context:20s}: {acc:6.2f}% {bar}")

print("\nACCURACY BY GRADE LEVEL:")
print("-" * 40)
grade_results = {
    'Elementary (G1-6)': results.get('G1-6', 0),
    'Secondary (G7-12)': results.get('G7-12', 0)
}
for grade, acc in grade_results.items():
    if acc > 0:
        print(f"{grade:20s}: {acc:6.2f}%")

print("\nDETAILED RESULTS TABLE:")
print("-" * 60)

results_data = []
for key, value in results.items():
    if key not in ['Overall', 'Valid_predictions']:
        results_data.append({'Metric': key, 'Accuracy (%)': f"{value:.2f}"})

df_results = pd.DataFrame(results_data)
print(df_results.to_string(index=False))

print("\nPREDICTION ANALYSIS:")
print("-" * 40)

valid_mask = preds >= 0
total_predictions = len(preds)
correct_predictions = (preds[valid_mask] == labels[valid_mask]).sum()
wrong_predictions = (preds[valid_mask] != labels[valid_mask]).sum()
failed_predictions = (~valid_mask).sum()

print(f"Total Questions:       {total_predictions:5d}")
print(f"Correct Predictions:   {correct_predictions:5d} ({correct_predictions/total_predictions*100:5.2f}%)")
print(f"Wrong Predictions:     {wrong_predictions:5d} ({wrong_predictions/total_predictions*100:5.2f}%)")
print(f"Failed Extractions:    {failed_predictions:5d} ({failed_predictions/total_predictions*100:5.2f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
subjects = list(subject_results.keys())
subject_accs = list(subject_results.values())
colors1 = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars1 = ax1.bar(subjects, subject_accs, color=colors1, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax1.set_title('Accuracy by Subject', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3)
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax2 = axes[0, 1]
contexts = list(context_results.keys())
context_accs = list(context_results.values())
colors2 = ['#95E1D3', '#F38181', '#AA96DA', '#FCBAD3']
bars2 = ax2.bar(contexts, context_accs, color=colors2, alpha=0.8, edgecolor='black')
ax2.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax2.set_title('Accuracy by Context Type', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', alpha=0.3)
for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax3 = axes[1, 0]
grades = list(grade_results.keys())
grade_accs = list(grade_results.values())
colors3 = ['#FFA07A', '#98D8C8']
bars3 = ax3.bar(grades, grade_accs, color=colors3, alpha=0.8, edgecolor='black')
ax3.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax3.set_title('Accuracy by Grade Level', fontsize=13, fontweight='bold')
ax3.set_ylim(0, 100)
ax3.grid(axis='y', alpha=0.3)
for bar in bars3:
    height = bar.get_height()
    if height > 0:
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax4 = axes[1, 1]
prediction_data = [correct_predictions, wrong_predictions, failed_predictions]
prediction_labels = ['Correct', 'Wrong', 'Failed']
colors4 = ['#90EE90', '#FFB6C6', '#FFD700']
explode = (0.05, 0.05, 0.05)
wedges, texts, autotexts = ax4.pie(prediction_data, labels=prediction_labels, autopct='%1.1f%%',
                                     colors=colors4, explode=explode, startangle=90,
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})
ax4.set_title('Prediction Breakdown', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'evaluation_results.png'")

print("\n" + "="*60)
print("SAMPLE PREDICTIONS (First 10 examples):")
print("="*60)

for i in range(min(10, len(preds))):
    pred_letter = chr(65 + preds[i]) if preds[i] >= 0 else "FAILED"
    true_letter = chr(65 + labels[i])
    status = "CORRECT" if preds[i] == labels[i] else "WRONG"

    print(f"\nExample {i+1}:")
    print(f"  Prediction: {pred_letter} | True Answer: {true_letter} | {status}")
    print(f"  Model Response: {responses[i][:100]}")

print("\n" + "="*60)
print("Evaluation Complete!")
print("="*60 + "\n")


# Finetuning

In [ ]:
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch
from dataclasses import dataclass
from typing import Dict, List
import warnings
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()
warnings.filterwarnings('ignore')

train_dataset_full = dataset['train']

TRAIN_SAMPLES = 5000
if TRAIN_SAMPLES:
    train_dataset_subset = train_dataset_full.select(range(TRAIN_SAMPLES))
    print(f"Training on {TRAIN_SAMPLES} samples")
else:
    train_dataset_subset = train_dataset_full
    print(f"Training on full dataset: {len(train_dataset_full)} samples")


In [ ]:
class ScienceQADataset(Dataset):
    """Custom dataset for ScienceQA - images only"""

    def __init__(self, hf_dataset, processor, model_type):
        self.processor = processor
        self.model_type = model_type

        print("Filtering dataset to only image-based questions...")
        self.dataset = [ex for ex in hf_dataset if ex['image'] is not None]
        print(f"Filtered dataset size: {len(self.dataset)} samples (with images)")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]

        question = example['question']
        choices = example['choices']
        hint = example['hint'] if example['hint'] else ""
        answer_idx = example['answer']
        image = example['image']

        prompt = create_prompt(question, choices, hint)
        answer_letter = chr(65 + answer_idx)

        if self.model_type == "llava":
            text_input = f"<image>\n{prompt}"
        else:
            text_input = prompt

        text_output = f" {answer_letter}"

        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        if image.mode != 'RGB':
            image = image.convert('RGB')

        return {
            'image': image,
            'text': text_input + text_output,
        }


In [ ]:
@dataclass
class MultimodalCollator:
    """Simple collator for image-only training"""

    processor: AutoProcessor

    def __call__(self, batch: List[Dict]) -> Dict[str, torch.Tensor]:
        images = [item['image'] for item in batch]
        texts = [item['text'] for item in batch]

        inputs = self.processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True,
        )

        labels = inputs["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        inputs["labels"] = labels

        return inputs


In [ ]:
val_dataset_full = dataset['validation'] if 'validation' in dataset else dataset['test']
VAL_SAMPLES = 500
val_dataset_subset = val_dataset_full.select(range(min(VAL_SAMPLES, len(val_dataset_full))))

print("Creating datasets...")
train_dataset_formatted = ScienceQADataset(train_dataset_subset, processor, MODEL_TYPE)
val_dataset_formatted = ScienceQADataset(val_dataset_subset, processor, MODEL_TYPE)

print(f"Training samples (with images): {len(train_dataset_formatted)}")
print(f"Validation samples (with images): {len(val_dataset_formatted)}")


In [ ]:
data_collator = MultimodalCollator(processor=processor)

training_args = TrainingArguments(
    output_dir="./scienceqa_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    dataloader_num_workers=0,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,
    data_collator=data_collator,
)


In [ ]:
# ==================== INSTALL PEFT ====================
!pip install peft -q

# ==================== SETUP LoRA ====================
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import gc

torch.cuda.empty_cache()
gc.collect()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model.enable_input_require_grads()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./scienceqa_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
trainer.train()

model.save_pretrained("./scienceqa_lora_final")
processor.save_pretrained("./scienceqa_lora_final")
print("LoRA adapters saved!")


In [ ]:
results_finetuned, responses_finetuned, preds_finetuned, labels_finetuned = evaluate_with_breakdown(test_dataset)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("\n" + "="*60)
print("EVALUATION RESULTS - ScienceQA Test Set (Finetuned)")
print("="*60)

print("\nOVERALL PERFORMANCE:")
print("-" * 40)
print(f"Overall Accuracy:      {results_finetuned['Overall']:.2f}%")
print(f"Valid Predictions:     {results_finetuned['Valid_predictions']:.2f}%")
print(f"Failed Extractions:    {100 - results_finetuned['Valid_predictions']:.2f}%")

print("\nACCURACY BY SUBJECT:")
print("-" * 40)
subject_results = {
    'Natural Science': results_finetuned.get('NAT', 0),
    'Social Science': results_finetuned.get('SOC', 0),
    'Language Science': results_finetuned.get('LAN', 0)
}
for subject, acc in subject_results.items():
    if acc > 0:
        print(f"{subject:20s}: {acc:6.2f}%")

print("\nACCURACY BY CONTEXT TYPE:")
print("-" * 40)
context_results = {
    'Text Only': results_finetuned.get('TXT', 0),
    'Image Only': results_finetuned.get('IMG', 0),
    'No Context': results_finetuned.get('NO', 0),
    'Text + Image': results_finetuned.get('TXT+IMG', 0)
}
for context, acc in context_results.items():
    if acc > 0:
        bar = '|' * int(acc / 2)
        print(f"{context:20s}: {acc:6.2f}% {bar}")

print("\nACCURACY BY GRADE LEVEL:")
print("-" * 40)
grade_results = {
    'Elementary (G1-6)': results_finetuned.get('G1-6', 0),
    'Secondary (G7-12)': results_finetuned.get('G7-12', 0)
}
for grade, acc in grade_results.items():
    if acc > 0:
        print(f"{grade:20s}: {acc:6.2f}%")

print("\nDETAILED RESULTS TABLE:")
print("-" * 60)

results_data = []
for key, value in results_finetuned.items():
    if key not in ['Overall', 'Valid_predictions']:
        results_data.append({'Metric': key, 'Accuracy (%)': f"{value:.2f}"})

df_results = pd.DataFrame(results_data)
print(df_results.to_string(index=False))

print("\nPREDICTION ANALYSIS:")
print("-" * 40)

valid_mask = preds_finetuned >= 0
total_predictions = len(preds_finetuned)
correct_predictions = (preds_finetuned[valid_mask] == labels_finetuned[valid_mask]).sum()
wrong_predictions = (preds_finetuned[valid_mask] != labels_finetuned[valid_mask]).sum()
failed_predictions = (~valid_mask).sum()

print(f"Total Questions:       {total_predictions:5d}")
print(f"Correct Predictions:   {correct_predictions:5d} ({correct_predictions/total_predictions*100:5.2f}%)")
print(f"Wrong Predictions:     {wrong_predictions:5d} ({wrong_predictions/total_predictions*100:5.2f}%)")
print(f"Failed Extractions:    {failed_predictions:5d} ({failed_predictions/total_predictions*100:5.2f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
bars1 = ax1.bar(list(subject_results.keys()), list(subject_results.values()),
                color=['#FF6B6B', '#4ECDC4', '#45B7D1'], alpha=0.8, edgecolor='black')
ax1.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax1.set_title('Accuracy by Subject', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3)
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax2 = axes[0, 1]
bars2 = ax2.bar(list(context_results.keys()), list(context_results.values()),
                color=['#95E1D3', '#F38181', '#AA96DA', '#FCBAD3'], alpha=0.8, edgecolor='black')
ax2.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax2.set_title('Accuracy by Context Type', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', alpha=0.3)
for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax3 = axes[1, 0]
bars3 = ax3.bar(list(grade_results.keys()), list(grade_results.values()),
                color=['#FFA07A', '#98D8C8'], alpha=0.8, edgecolor='black')
ax3.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax3.set_title('Accuracy by Grade Level', fontsize=13, fontweight='bold')
ax3.set_ylim(0, 100)
ax3.grid(axis='y', alpha=0.3)
for bar in bars3:
    height = bar.get_height()
    if height > 0:
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax4 = axes[1, 1]
wedges, texts, autotexts = ax4.pie(
    [correct_predictions, wrong_predictions, failed_predictions],
    labels=['Correct', 'Wrong', 'Failed'],
    autopct='%1.1f%%',
    colors=['#90EE90', '#FFB6C6', '#FFD700'],
    explode=(0.05, 0.05, 0.05),
    startangle=90,
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)
ax4.set_title('Prediction Breakdown', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_results_finetuned.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'evaluation_results_finetuned.png'")

print("\n" + "="*60)
print("SAMPLE PREDICTIONS (First 10 examples):")
print("="*60)

for i in range(min(10, len(preds_finetuned))):
    pred_letter = chr(65 + preds_finetuned[i]) if preds_finetuned[i] >= 0 else "FAILED"
    true_letter = chr(65 + labels_finetuned[i])
    status = "CORRECT" if preds_finetuned[i] == labels_finetuned[i] else "WRONG"

    print(f"\nExample {i+1}:")
    print(f"  Prediction: {pred_letter} | True Answer: {true_letter} | {status}")
    print(f"  Model Response: {responses_finetuned[i][:100]}")

print("\n" + "="*60)
print("Evaluation Complete!")
print("="*60 + "\n")


In [ ]:
print("\n" + "="*50)
print("COMPARISON: Before vs After Fine-tuning")
print("="*50)

comparison_df = []
for key in results.keys():
    if key in results_finetuned:
        comparison_df.append({
            'Metric': key,
            'Before (%)': f"{results[key]:.2f}",
            'After (%)': f"{results_finetuned[key]:.2f}",
            'Improvement': f"{results_finetuned[key] - results[key]:+.2f}"
        })

import pandas as pd
comparison_table = pd.DataFrame(comparison_df)
print("\n", comparison_table.to_string(index=False))

print("\n" + "="*50)
print(f"Overall improvement: {results_finetuned['Overall'] - results['Overall']:+.2f}%")
print("="*50)
